# Music Store Customer Support — Multi-Agent System

Customer support workflow using **LangChain**, **LangGraph**, and the **Chinook** SQLite music store database.

**Features**
- Customer verification (ID / email / phone) with human-in-the-loop interrupts
- Music Catalog sub-agent
- Invoice Information sub-agent
- Supervisor routing (music / invoice / both)
- Short-term session memory (`customer_id`) + long-term preference memory

## 1. Setup

Set your Groq API key below (or via a `GROQ_API_KEY` environment variable).

In [2]:
import os
from getpass import getpass

# Prefer env var; otherwise prompt securely
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

API_KEY = os.environ["GROQ_API_KEY"]
print("API key loaded.")

API key loaded.


## 2. Chinook database

Download and load the Chinook SQL script into an in-memory SQLite database.

In [3]:
from db import get_db, run_query

db = get_db()
print("Tables:", db.get_usable_table_names())
print("Sample customers:")
print(run_query("SELECT CustomerId, FirstName, LastName, Email, Phone FROM Customer LIMIT 5;"))

Tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample customers:
[(1, 'Luís', 'Gonçalves', 'luisg@embraer.com.br', '+55 (12) 3923-5555'), (2, 'Leonie', 'Köhler', 'leonekohler@surfeu.de', '+49 0711 2842222'), (3, 'François', 'Tremblay', 'ftremblay@gmail.com', '+1 (514) 721-4711'), (4, 'Bjørn', 'Hansen', 'bjorn.hansen@yahoo.no', '+47 22 44 22 22'), (5, 'František', 'Wichterlová', 'frantisekw@jetbrains.com', '+420 2 4172 5555')]


## 3. Tools (quick smoke test)

In [4]:
from tools_music import get_albums_by_artist, get_tracks_by_artist
from tools_invoice import get_invoices_by_customer_sorted_by_date

print("Albums by Rolling Stones:")
print(get_albums_by_artist.invoke({"artist": "Rolling Stones"}))
print("\nRecent invoices for customer 1:")
print(get_invoices_by_customer_sorted_by_date.invoke({"customer_id": "1"}))

Albums by Rolling Stones:
[('Hot Rocks, 1964-1971 (Disc 1)', 'The Rolling Stones'), ('No Security', 'The Rolling Stones'), ('Voodoo Lounge', 'The Rolling Stones')]

Recent invoices for customer 1:
[(382, '2025-08-07 00:00:00', 8.91, 'São José dos Campos', 'Brazil'), (327, '2024-12-07 00:00:00', 13.86, 'São José dos Campos', 'Brazil'), (316, '2024-10-27 00:00:00', 1.98, 'São José dos Campos', 'Brazil'), (195, '2023-05-06 00:00:00', 0.99, 'São José dos Campos', 'Brazil'), (143, '2022-09-15 00:00:00', 5.94, 'São José dos Campos', 'Brazil'), (121, '2022-06-13 00:00:00', 3.96, 'São José dos Campos', 'Brazil'), (98, '2022-03-11 00:00:00', 3.98, 'São José dos Campos', 'Brazil')]


## 4. Build the multi-agent graph

State schema: `customer_id`, `messages`, `loaded_memory`, `route`.

Flow: **verify → load memory → supervisor → (music | invoice | both) → save memory**

In [5]:
from memory_store import preference_memory
from graph import ask, build_graph, resume

preference_memory.clear()
graph = build_graph(API_KEY)
print("Graph compiled.")

Graph compiled.


## 5. Test case 1 — providing customer information

Phone is included, so verification should succeed without an interrupt.
Supervisor should route to **both** (invoice + music).

In [6]:
THREAD_1 = "testcase-1"

q1 = (
    "My phone number is +55 (12) 3923-5555. "
    "How much was my most recent purchase? "
    "What albums do you have by the Rolling Stones?"
)

result1 = ask(graph, q1, thread_id=THREAD_1)
print("Status:", result1["status"])
print("Customer ID:", result1.get("customer_id"))
print("Memory:", result1.get("loaded_memory"))
print("\nAnswer:\n", result1.get("answer") or result1.get("prompt"))

Status: ok
Customer ID: 1
Memory: Rolling Stones

Answer:
 I've found the information you requested:

Your most recent purchase was on '2025-08-07' and the total was $8.91.

We have the following albums by the Rolling Stones: 
1. Hot Rocks, 1964-1971 (Disc 1)
2. No Security
3. Voodoo Lounge


### Follow-up — preferences from long-term memory

Same session. Preferences should include Rolling Stones from the previous turn.

In [7]:
q1b = "List some songs that match my preferences?"
result1b = ask(graph, q1b, thread_id=THREAD_1)
print("Status:", result1b["status"])
print("Customer ID:", result1b.get("customer_id"))
print("Memory:", result1b.get("loaded_memory"))
print("\nAnswer:\n", result1b.get("answer") or result1b.get("prompt"))

Status: ok
Customer ID: 1
Memory: Rolling Stones

Answer:
 Based on the available data, here are some songs by The Rolling Stones that you might enjoy: 
1. '19th Nervous Breakdown' 
2. 'Paint It Black' 
3. 'Get Off Of My Cloud' 
4. 'As Tears Go By' 
5. 'Out Of Tears' 

These songs are from different albums, including 'Hot Rocks, 1964-1971' and 'Voodoo Lounge'.


## 6. Test case 2 — missing customer information (new session)

No credentials → workflow **interrupts** and asks for Customer ID / email / phone.

In [8]:
THREAD_2 = "testcase-2"

q2 = (
    "How much was my most recent purchase? "
    "What albums do you have by the Rolling Stones?"
)

result2 = ask(graph, q2, thread_id=THREAD_2)
print("Status:", result2["status"])
print(result2.get("prompt") or result2.get("answer"))

Status: interrupted
{'prompt': 'I need to verify your identity before answering account questions. Please provide your Customer ID, email, or phone number.'}


### Resume with credentials (human-in-the-loop)

Provide a valid Chinook phone / email / customer id to continue.

In [9]:
# Example: resume with the same Brazilian phone used in test case 1
result2b = resume(graph, "+55 (12) 3923-5555", thread_id=THREAD_2)
print("Status:", result2b["status"])
print("Customer ID:", result2b.get("customer_id"))
print("\nAnswer:\n", result2b.get("answer") or result2b.get("prompt"))

Status: ok
Customer ID: 1

Answer:
 I've found the information you requested:

Your most recent purchase was on '2025-08-07' and the total was $8.91.

We have the following albums by the Rolling Stones: 
1. Hot Rocks, 1964-1971 (Disc 1)
2. No Security
3. Voodoo Lounge


## 7. Extra checks

In [10]:
# Music-only (no identity required)
r = ask(graph, "Do you have any Metallica albums?", thread_id="music-only")
print(r.get("answer") or r)

# Show stored preferences
print("\nPreference store:", preference_memory._store)

Yes, we have Metallica albums. Our catalog includes:

1. ...And Justice For All
2. Black Album
3. Garage Inc. (Disc 1)
4. Garage Inc. (Disc 2)
5. Kill 'Em All
6. Load
7. Master Of Puppets
8. ReLoad
9. Ride The Lightning
10. St. Anger

You can choose from these titles.

Preference store: {'1': 'Rolling Stones'}
